# agents

> This module specifies the Agent, Policy, and RL Classes. An agent carries one or more policies when exploring an environment, and updates parameters using reinforcement learning.

In [ ]:
#| default_exp agents

In [ ]:
#| export
import numpy as np

In [ ]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
class Policy:
    '''This Policy Class specifies a general architecture. '''
    def __init__(self):
        self.theta = None
        self.actions = None
        self.pi = None

In [ ]:
#| export
class flexiblePolicy(Policy):
    '''This flexiblePolicy Class specifies a fully flexible policy with a set of basis functions
    and a learnable set of weights. '''
    def __init__(self, omega=None, fk=None, wParams=None, integ=0.0):
        super().__init__()
        self.omega = np.asarray(omega) if omega is not None else None
        self.fk = np.asarray(fk) if fk is not None else None
        self.wParams = wParams
        self.integ = float(integ)

In [ ]:
#| export
class RL:
    '''This RL Class performs reinforcement learning: after taking an action, update parameters
    based on feedback from the environment.
    '''
    def __init__(self,policy):
        self.policy = policy

    def chooseAction(self):
        raise NotImplementedError

    def updateParams(self):
        raise NotImplementedError

In [ ]:
#| export
class Agent:
    '''This Class specifies agents with a defined policy and reinforcement learning algorithm. '''
    def __init__(self,theta,goal,policy,rl,params):
        self.theta = theta   # current heading (radians)
        self.goal = goal
        self.policy = policy
        self.rl = rl
        self.params = params  # structure containing simulation parameters
        self.dx = None  # change in heading
        self.dt = None  # change in time

    def samplePolicy(self):
        raise NotImplementedError("Override me")

In [ ]:
#| export
class flexibleAgent(Agent):
    '''Agent using a flexiblePolicy and reinforcement learning algorithm. '''
    def __init__(self, theta, goal, policy, rl, params):
        if not isinstance(policy, flexiblePolicy):
            raise TypeError('policy must be a flexiblePolicy instance')
        if policy.omega is None or policy.fk is None or policy.wParams is None:
            raise ValueError('policy needs omega, fk and wParams')
        nb = policy.omega.size // 2
        if policy.omega.size != 2 * nb:
            raise ValueError('omega must have even length')
        if policy.fk.shape != (nb, params.nx):
            raise ValueError(f'fk must have shape (omega.size // 2, params.nx) = {(nb, params.nx)}')
        if len(policy.wParams) < 6:
            raise ValueError('wParams needs 6 values: (k, fmax, offset) for drift, then for turn bias')
        super().__init__(theta, goal, policy, rl, params)

    def _heading_index(self, nx):
        centers = np.linspace(0, 2 * np.pi, num=nx, endpoint=False)
        # wrapped angular distance, valid for any theta (negative or beyond 2*pi)
        d = np.abs((self.theta - centers + np.pi) % (2 * np.pi) - np.pi)
        return int(np.argmin(d))

    def samplePolicy(self):
        '''samples a change in heading over a change in time to use in training'''
        pol = self.policy
        nx = self.params.nx
        ind = self._heading_index(nx)

        # the drift-diffusion integrator lives on the policy
        nu, pR = self.convertParams(pol.omega, pol.fk[:, ind], pol.wParams)
        pol.integ = pol.integ + np.random.normal(
            nu * self.params.delT, self.params.sigF * np.sqrt(self.params.delT)
        )

        if pol.integ > self.params.aF:
            if self.flipCoin(pR):
                direction = 1
            else:
                direction = -1
            self.dx = np.multiply(
                np.multiply(direction, np.random.lognormal(self.params.muS, self.params.sigmaS)),
                96.0 / 360,
            )
            self.dt = self.params.dtS
            pol.integ = 0.0
        else:
            self.dx = 0
            self.dt = 1

        self.dx = round(self.dx)
        self.dt = round(self.dt)
        return self.dx, self.dt, pol.integ

    def convertParams(self, omega, fk_col, w_params):
        '''converts a vector of weights into drift rate and turn bias (scalars)'''
        omega = np.asarray(omega)
        fk_col = np.asarray(fk_col)
        nb = omega.size // 2
        if omega.size != 2 * nb:
            raise ValueError('omega must have even length')
        if fk_col.size != nb:
            raise ValueError('fk column length must match omega.size // 2')
        omega_f, omega_r = omega[:nb], omega[nb:]
        nu_f = self.sigmoid(omega_f @ fk_col, w_params[0], w_params[1], w_params[2])
        p_r = self.sigmoid(omega_r @ fk_col, w_params[3], w_params[4], w_params[5])
        return float(nu_f), float(p_r)

    def sigmoid(self, x, k, fmax, offset):
        '''FMAX/(1+exp(-K*X))-OFFSET'''
        return np.divide(fmax, (1 + np.exp(-k * x))) - offset

    def flipCoin(self, p):
        '''samples from a Bernoulli process with probability p'''
        u = np.random.rand()
        return u < p

In [ ]:
from types import SimpleNamespace
from fastcore.test import test_eq, test_fail

def toy_agent(theta=0.0, aF=10.0, nb=4, nx=8, omega=None):
    "Builds a small flexibleAgent for testing."
    policy = flexiblePolicy(omega=np.zeros(2 * nb) if omega is None else omega,
                            fk=np.ones((nb, nx)),
                            wParams=np.array([0.5, 1.0, 0.0, 0.5, 1.0, 0.0]))
    params = SimpleNamespace(nx=nx, delT=0.01, sigF=0.1, aF=aF, muS=0.0, sigmaS=0.5, dtS=5)
    return flexibleAgent(theta, None, policy, RL(policy), params)

`convertParams` returns one drift rate and one turn probability:

In [ ]:
agent = toy_agent()
nu, pR = agent.convertParams(agent.policy.omega, agent.policy.fk[:, 0], agent.policy.wParams)
assert np.isscalar(nu) and np.isscalar(pR)
test_eq(nu, 0.5)   # omega is all zeros, so sigmoid(0) = fmax/2
test_eq(pR, 0.5)

With a high threshold the agent fixates: no turn, one time step, and the integrator keeps its value:

In [ ]:
agent = toy_agent(aF=10.0)
dx, dt, integ = agent.samplePolicy()
test_eq((dx, dt), (0, 1))
test_eq(integ, agent.policy.integ)
assert integ != 0.0

With a threshold below zero the agent always saccades: the step lasts `dtS` and the integrator resets:

In [ ]:
agent = toy_agent(aF=-1.0)
dx, dt, integ = agent.samplePolicy()
test_eq(dt, 5)
test_eq(integ, 0.0)
test_eq(agent.policy.integ, 0.0)
assert isinstance(dx, (int, np.integer))

Heading bins wrap around the circle, so a heading just below 2π (or just below 0) maps to bin 0, and headings beyond 2π wrap back:

In [ ]:
test_eq(toy_agent(theta=2 * np.pi - 0.01)._heading_index(8), 0)
test_eq(toy_agent(theta=np.pi)._heading_index(8), 4)
test_eq(toy_agent(theta=-0.01)._heading_index(8), 0)
test_eq(toy_agent(theta=7 * np.pi)._heading_index(8), 4)

Bad inputs fail early with a clear error:

In [ ]:
test_fail(lambda: toy_agent(omega=np.zeros(7)), contains='even length')
test_fail(lambda: flexibleAgent(0.0, None, Policy(), None, SimpleNamespace(nx=8)), contains='flexiblePolicy')
test_fail(lambda: RL(None).chooseAction())
test_fail(lambda: RL(None).updateParams())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()